Trigger dei workflow (on: push, on: pull_request, on: schedule...) — capire come questi sistemi AI vengono integrati e testati

In [5]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter, defaultdict

DATASET_PATH = "/Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/gigawork/dataset_with_ids.csv"
BASE_GIGAWORK_PATH = "/Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/gigawork/all_workflows"

df = pd.read_csv(DATASET_PATH)

In [10]:
import yaml
from pathlib import Path
from collections import Counter


def get_on_section(workflow_dict):
    """Ritorna il contenuto della chiave `on`, gestendo anche il parsing YAML 1.1 (`on` -> True)."""
    if not isinstance(workflow_dict, dict):
        return None

    if "on" in workflow_dict:
        return workflow_dict["on"]

    # In PyYAML, la chiave `on` non quotata puo essere interpretata come boolean True.
    if True in workflow_dict:
        return workflow_dict[True]

    for k, v in workflow_dict.items():
        if isinstance(k, str) and k.strip().lower() == "on":
            return v

    return None


def extract_triggers(on_value):
    """Estrae i trigger principali dal campo `on` di una workflow GitHub Actions."""
    if on_value is None:
        return []

    if isinstance(on_value, str):
        return [on_value.strip()] if on_value.strip() else []

    if isinstance(on_value, list):
        triggers = []
        for item in on_value:
            if isinstance(item, str) and item.strip():
                triggers.append(item.strip())
            elif isinstance(item, dict):
                triggers.extend([str(k).strip() for k in item.keys() if str(k).strip()])
        return sorted(set(triggers))

    if isinstance(on_value, dict):
        return sorted({str(k).strip() for k in on_value.keys() if str(k).strip()})

    return []


stats_per_repo = {}
total_trigger_counter = Counter()
total_trigger_counter_unique = Counter()

for repo in repo_summary["repository"].unique():
    files = repo_summary.loc[repo_summary["repository"] == repo, "files"].values[0]
    repo_triggers = Counter()

    for file in files:
        path = Path(BASE_GIGAWORK_PATH) / repo / file
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
        except Exception:
            continue

        on_section = get_on_section(data)
        triggers = extract_triggers(on_section)

        if not triggers:
            continue

        repo_triggers.update(triggers)
        total_trigger_counter.update(triggers)
        total_trigger_counter_unique.update(set(triggers))

    stats_per_repo[repo] = dict(repo_triggers)

print("Top trigger totali:", total_trigger_counter.most_common(20))
print("Top trigger unici per file:", total_trigger_counter_unique.most_common(20))



Top trigger totali: [('push', 307), ('pull_request', 280), ('workflow_dispatch', 213), ('schedule', 74), ('workflow_call', 55), ('pull_request_target', 51), ('release', 39), ('issues', 19), ('workflow_run', 18), ('issue_comment', 17), ('merge_group', 16), ('pull_request_review', 8), ('repository_dispatch', 5), ('pull_request_review_comment', 5), ('branch_protection_rule', 2), ('delete', 1)]
Top trigger unici per file: [('push', 307), ('pull_request', 280), ('workflow_dispatch', 213), ('schedule', 74), ('workflow_call', 55), ('pull_request_target', 51), ('release', 39), ('issues', 19), ('workflow_run', 18), ('issue_comment', 17), ('merge_group', 16), ('pull_request_review', 8), ('repository_dispatch', 5), ('pull_request_review_comment', 5), ('branch_protection_rule', 2), ('delete', 1)]
Trigger estratti da esempio: ['pull_request']


In [13]:
from collections import Counter, defaultdict
from pathlib import Path
import yaml


def extract_job_keywords(workflow_dict):
    """Estrae le chiavi top-level della sezione jobs (es. tests, build, deploy)."""
    if not isinstance(workflow_dict, dict):
        return []

    jobs_section = workflow_dict.get("jobs")
    if isinstance(jobs_section, dict):
        return [str(k).strip() for k in jobs_section.keys() if str(k).strip()]

    return []


# trigger -> {total: int, jobs_counter: Counter}
trigger_stats = defaultdict(lambda: {"total": 0, "jobs_counter": Counter()})

for repo in repo_summary["repository"].unique():
    files = repo_summary.loc[repo_summary["repository"] == repo, "files"].values[0]

    for file in files:
        path = Path(BASE_GIGAWORK_PATH) / repo / file

        try:
            with open(path, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
        except Exception:
            continue

        if not isinstance(data, dict):
            continue

        triggers = extract_triggers(get_on_section(data))
        if not triggers:
            continue

        job_keywords = extract_job_keywords(data)

        # Conta una volta per trigger per file workflow
        for trigger in set(triggers):
            trigger_stats[trigger]["total"] += 1
            trigger_stats[trigger]["jobs_counter"].update(job_keywords)


result = {
    trigger: {
        "total": values["total"],
        "most_associated_keywords": dict(values["jobs_counter"].most_common(20)),
    }
    for trigger, values in sorted(
        trigger_stats.items(),
        key=lambda item: item[1]["total"],
        reverse=True,
    )
}

print(result)
import json
print(json.dumps(result,sort_keys=True,indent=2))


{'push': {'total': 307, 'most_associated_keywords': {'build': 66, 'test': 38, 'deploy': 20, 'release': 16, 'lint': 15, 'analyze': 11, 'publish': 9, 'create-release': 5, 'pre-commit': 4, 'pytest': 4, 'migrate': 4, 'conflicts': 3, 'size': 3, 'notify': 3, 'setup': 3, 'check': 3, 'docs': 3, 'require-all-checks-to-pass': 3, 'publish_docker_to_ecr': 3, 'codespell': 2}}, 'pull_request': {'total': 280, 'most_associated_keywords': {'build': 37, 'test': 37, 'lint': 17, 'analyze': 10, 'pytest': 5, 'format': 5, 'pre-commit': 4, 'changed-files': 4, 'main': 4, 'deploy': 3, 'setup': 3, 'claude-review': 3, 'type-check': 3, 'mypy': 3, 'require-all-checks-to-pass': 3, 'pyright': 3, 'unit-tests': 3, 'changes': 3, 'verify-version': 3, 'block-prerelease': 3}}, 'workflow_dispatch': {'total': 213, 'most_associated_keywords': {'build': 31, 'deploy': 20, 'publish': 15, 'release': 10, 'test': 7, 'lint': 6, 'create-release': 5, 'build-image': 4, 'merge': 4, 'docker': 4, 'stale': 3, 'e2e': 3, 'e2e-app-store': 3, 